# 4 — Feature Engineering

Derives chunk-level and session-level features from Pass 1 (Gemini) annotation outputs.

**Inputs:**
- `data/chunk_registry_v1.csv` — master chunk index with outcome labels
- `outputs/<conf>/output_<session>/<chunk>.json` — one per chunk, from Pass 1

**Outputs:**
- `data/chunk_features.parquet` — one row per chunk, all chunk-level derived features
- `data/session_features.parquet` — one row per session_group, aggregated + delta features
- `data/model_ready_features.parquet` — session features after VIF filter, joined with outcomes
- `data/feature_manifest.csv` — feature name and inclusion flag

All feature engineering is deterministic and version-controlled per `research_project_plan_v2.md` Stage 4.

## Step 0 — Imports and paths

In [52]:
import json
import re
import warnings
from pathlib import Path
from statistics import mean

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# ── Resolve directories relative to this notebook ────────────────────────────
NOTEBOOK_DIR = Path().resolve()                      # analysis_v2/notebooks/
ANALYSIS_V2  = NOTEBOOK_DIR.parent                   # analysis_v2/
GEMINI_CODE  = ANALYSIS_V2.parent                    # NICO/gemini_code/
DATA_DIR     = ANALYSIS_V2 / 'data'
OUTPUT_DIR   = GEMINI_CODE / 'outputs'

DATA_DIR.mkdir(parents=True, exist_ok=True)

EPSILON = 1e-6   # division guard

print('GEMINI_CODE:', GEMINI_CODE)
print('DATA_DIR   :', DATA_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)

GEMINI_CODE: /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis
DATA_DIR   : /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data
OUTPUT_DIR : /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/outputs


## Step 1 — Load chunk registry and resolve JSON paths

In [53]:
registry = pd.read_csv(DATA_DIR / 'chunk_registry_v1.csv')
print(f'Registry: {len(registry):,} chunks from {registry["session_key"].nunique():,} recordings '
      f'across {registry["conference_id"].nunique()} conferences')
registry.head(2)

Registry: 1,310 chunks from 196 recordings across 8 conferences


,chunk_id,conference_id,session_key,session_group,chunk_file_name,chunk_path,gemini_file_ref,chunk_index,n_chunks_in_session,chunk_position,analyzed,model_used,num_teams,num_funded_teams,has_teams,has_funded_teams,human_validation_set,utterance_validation_set,oversampled_for,outcome_has_funded_teams
0,2020NES__2020_11_05_NES_S5/Zoom_Meeting_11_5_2...,2020NES,2020_11_05_NES_S5/Zoom_Meeting_11_5_2020_12_17...,2020_11_05_NES_S5,Zoom_Meeting_11_5_2020_12_17_50_PM.mp4,/home/jhs0727/p32824/scialog/2020NES/2020_11_0...,files/v29biz03mi07,0,1,whole,True,gemini-3.1-pro-preview,1.0,1.0,True,True,False,False,NaN,True
1,2020NES__2020_11_05_NES_S5/Zoom_Meeting_11_5_2...,2020NES,2020_11_05_NES_S5/Zoom_Meeting_11_5_2020_12_11...,2020_11_05_NES_S5,Zoom_Meeting_11_5_2020_12_11_13_PM.mp4,/home/jhs0727/p32824/scialog/2020NES/2020_11_0...,files/3ps3pbnz8s7j,0,1,whole,True,gemini-3.1-pro-preview,1.0,1.0,True,True,False,False,NaN,True


In [54]:
def normalize_filename(name: str) -> str:
    """Normalize a video filename stem to match the output JSON filename.
    
    Rules observed across conferences:
      - spaces, hyphens and similar chars → underscore
      - dots and parentheses are preserved
      - consecutive underscores collapsed to one
    """
    name = re.sub(r'[^a-zA-Z0-9._()]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def resolve_json_path(row: pd.Series) -> Path:
    """Return the Path to the Pass-1 output JSON for a registry row.
    
    Some output directories prepend 'ATTN_' to files that required attention
    during annotation; we fall back to that variant if the plain name is absent.
    Session groups stored with hyphens (e.g. 2021-05-20-ABI-S1) map to
    underscore-named output directories (output_2021_05_20_ABI_S1).
    """
    conf        = row['conference_id']
    session_key = row['session_key']
    chunk_fn    = row['chunk_file_name']

    stem_norm    = normalize_filename(Path(chunk_fn).stem)
    session_grp  = session_key.split('/')[0].replace('-', '_')
    out_dir      = OUTPUT_DIR / conf / f'output_{session_grp}'

    if '/' in session_key:
        recording_folder = session_key.split('/', 1)[1]
        base = out_dir / recording_folder
    else:
        base = out_dir

    p = base / f'{stem_norm}.json'
    if not p.exists():
        p_attn = base / f'ATTN_{stem_norm}.json'
        if p_attn.exists():
            return p_attn
    return p


# Resolve all paths and load JSONs
json_cache: dict[str, dict] = {}
missing_json = []

for _, row in registry.iterrows():
    chunk_id = row['chunk_id']
    path     = resolve_json_path(row)
    if path.exists():
        try:
            with open(path) as f:
                json_cache[chunk_id] = json.load(f)
        except (json.JSONDecodeError, Exception) as e:
            missing_json.append((chunk_id, str(e)[:60]))
    else:
        missing_json.append((chunk_id, 'file not found'))

print(f'Loaded: {len(json_cache):,} JSONs')
if missing_json:
    print(f'Skipped ({len(missing_json)} chunks — empty/malformed/missing):')
    for cid, reason in missing_json:
        print(f'  {cid}: {reason}')

Loaded: 1,286 JSONs
Skipped (24 chunks — empty/malformed/missing):
  2020NES__2020_11_06_NES_S6/6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55__6_CO2_Reduction_Zoom_Meeting_2020_11_06_08_45_55.mp4: Expecting value: line 1 column 1 (char 0)
  2021MND__2021_04_22_MND_S8__Zoom_Meeting_Room_6_2021_04_22_13_18_39_chunk1.mp4: Expecting value: line 1 column 1 (char 0)
  2021MND__2021_04_22_MND_S8__Zoom_Meeting_Room_6_2021_04_22_13_00_55_chunk2.mp4: Expecting value: line 1 column 1 (char 0)
  2021MND__2021_04_23_MND_S8__Zoom_Meeting_Room_6_2021_04_23_12_09_42_chunk2.mp4: Expecting value: line 1 column 1 (char 0)
  2021MND__2021_04_23_MND_S11__bot2_Zoom_Meeting_2021_04_23_13_13_56_chunk4.mp4: Expecting value: line 1 column 1 (char 0)
  2021MND__2021_04_23_MND_S13__botB1_2021_04_23_13_14_18_chunk3.mp4: Expecting value: line 1 column 1 (char 0)
  2021NES__2021-11-05-NES-S9__B3 - 2021-11-05 13-06-59_chunk3.mp4: Expecting value: line 1 column 1 (char 0)
  2021NES__2021-11-05-NES-S10__B4 - 2021-11

## Step 2 — Code name mapping

The 16 annotation categories use natural-language names in the JSON.  
We map them to the canonical underscore names used throughout Stage 4.

In [55]:
# ── Canonical code name → all accepted raw JSON name variants ────────────────
CODE_NAME_VARIANTS: dict[str, list[str]] = {
    'Idea_Management':          ['Idea Management'],
    'Information_Seeking':      ['Information Seeking'],
    'Knowledge_Sharing':        ['Knowledge Sharing'],
    'Evaluation_Practices':     ['Evaluation Practices'],
    'Relational_Climate':       ['Relational Climate'],
    'Participation_Dynamics':   ['Participation Dynamics'],
    'Coordination_Decision':    ['Coordination and Decision Practices'],
    'Integration_Practices':    ['Integration Practices'],
    'Idea_Ownership_Attribution': ['Idea Ownership and Attribution', 'Idea Ownership'],
    'Future_Oriented_Language': ['Future-Oriented Language'],
    'Epistemic_Bridging':       ['Epistemic Bridging'],
    'Pronoun_Framing':          ['Pronoun Framing'],
    'Complementarity_Articulation': ['Complementarity Articulation'],
    'Role_Anticipation':        ['Role Anticipation'],
    'Broader_Significance':     ['Broader Significance'],
    'Idea_Novelty_Signal':      ['Idea Novelty Signal'],
}

# Reverse: raw JSON code_name → canonical key
RAW_TO_CANONICAL: dict[str, str] = {
    raw: canonical
    for canonical, variants in CODE_NAME_VARIANTS.items()
    for raw in variants
}

ALL_CATEGORIES = list(CODE_NAME_VARIANTS.keys())
print('Canonical categories:')
for c in ALL_CATEGORIES:
    print(f'  {c}')

Canonical categories:
  Idea_Management
  Information_Seeking
  Knowledge_Sharing
  Evaluation_Practices
  Relational_Climate
  Participation_Dynamics
  Coordination_Decision
  Integration_Practices
  Idea_Ownership_Attribution
  Future_Oriented_Language
  Epistemic_Bridging
  Pronoun_Framing
  Complementarity_Articulation
  Role_Anticipation
  Broader_Significance
  Idea_Novelty_Signal


## Step 3 — Helper functions

In [56]:
# ── Gini coefficient ─────────────────────────────────────────────────────────
def compute_gini(values: list[float]) -> float:
    """Gini coefficient of speaking time distribution. 0 = perfectly equal."""
    if not values or sum(values) == 0:
        return 0.0
    arr = sorted(values)
    n   = len(arr)
    s   = sum(arr)
    cumsum = 0.0
    for i, v in enumerate(arr):
        cumsum += v
    # Standard formula: G = (2 * sum_i (i * x_i)) / (n * sum_i x_i) - (n+1)/n
    total = sum((i + 1) * v for i, v in enumerate(arr))
    return (2 * total) / (n * s) - (n + 1) / n


# ── Utterance code counting ──────────────────────────────────────────────────
def iter_codes(utterances: list[dict], canonical_category: str):
    """Yield code objects matching canonical_category across all utterances."""
    variants = set(CODE_NAME_VARIANTS.get(canonical_category, [canonical_category]))
    for u in utterances:
        for c in u.get('codes', []):
            if c.get('code_name') in variants:
                yield c


def count_codes(
    utterances: list[dict],
    canonical_category: str,
    subcodes: list[str] | None = None
) -> int:
    """Count code occurrences. Optionally filter to specific subcodes."""
    n = 0
    for c in iter_codes(utterances, canonical_category):
        if subcodes is None or c.get('subcode') in subcodes:
            n += 1
    return n


def count_interruption_type(utterances: list[dict], itype: str) -> int:
    """Count utterances with a specific interruption_type value."""
    return sum(1 for u in utterances if u.get('interruption_type') == itype)


# ── Idea quality helpers ─────────────────────────────────────────────────────
def _collect_idea_quality(utterances: list[dict], canonical_category: str) -> list[int]:
    """Collect non-null idea_quality scores for the given category."""
    return [
        c['idea_quality']
        for c in iter_codes(utterances, canonical_category)
        if 'idea_quality' in c and c['idea_quality'] is not None
    ]


def sum_all_scores(utterances: list[dict]) -> float:
    """Sum all idea_quality scores across the three quality-coded categories."""
    total = 0.0
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        total += sum(_collect_idea_quality(utterances, cat))
    return total


def mean_all_scores(utterances: list[dict]) -> float | None:
    """Mean idea_quality across all quality-coded categories."""
    scores = []
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores.extend(_collect_idea_quality(utterances, cat))
    return float(np.mean(scores)) if scores else None


def pct_scores_gte(utterances: list[dict], threshold: int) -> float | None:
    """Fraction of idea_quality scores >= threshold across quality-coded categories."""
    scores = []
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores.extend(_collect_idea_quality(utterances, cat))
    return sum(s >= threshold for s in scores) / len(scores) if scores else None


# ── Utterance-level boolean helpers ─────────────────────────────────────────
def strip_low_confidence(value):
    """Normalize annotation values like 'Yes [low_confidence]' to 'Yes'."""
    if isinstance(value, str):
        return value.replace('[low_confidence]', '').strip()
    return value


def coerce_numeric(value) -> float | None:
    """Parse numeric fields that may arrive as strings from the annotator."""
    value = strip_low_confidence(value)
    if value in (None, ''):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def pct_turns_with(utterances: list[dict], field: str, value) -> float:
    """Fraction of turns where utterance[field] matches value after normalization."""
    if not utterances:
        return 0.0
    target = strip_low_confidence(value)
    return sum(
        1 for u in utterances
        if strip_low_confidence(u.get(field)) == target
    ) / len(utterances)


def mean_field(utterances: list[dict], field: str) -> float | None:
    """Mean of a numeric utterance field after coercing annotation strings."""
    vals = [coerce_numeric(u.get(field)) for u in utterances if field in u]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None



# ── Z-score composite ────────────────────────────────────────────────────────
def mean_of_zscored(*series_values) -> float | None:
    """Z-score each non-None series element and average.
    
    Accepts scalar values (session-level use) or lists (cross-session use).
    Returns None if fewer than 2 non-None inputs.
    """
    vals = [v for v in series_values if v is not None and not np.isnan(float(v))]
    if len(vals) < 2:
        return None
    arr  = np.array(vals, dtype=float)
    std  = arr.std()
    if std == 0:
        return 0.0
    return float(((arr - arr.mean()) / std).mean())


# ── Session-level composite helpers ─────────────────────────────────────────
def safe_mean(values) -> float | None:
    """Mean of non-null values; None if no valid values."""
    vals = [v for v in values if v is not None and not (isinstance(v, float) and np.isnan(v))]
    return float(np.mean(vals)) if vals else None


print('Helper functions defined.')

Helper functions defined.


## Step 4 — Compute chunk-level features

### 4.1  Chunk-level derived features (from `chunk_summary`)
### 4.2  Utterance-level aggregated features  
### 4.3  Multimodal signal aggregates

In [57]:
chunk_rows = []

for _, reg_row in registry.iterrows():
    chunk_id = reg_row['chunk_id']
    data     = json_cache.get(chunk_id)

    if data is None:
        # JSON not found — emit a row of NaNs so the chunk is still present
        chunk_rows.append({'chunk_id': chunk_id, '_json_loaded': False})
        continue

    cs  = data.get('chunk_summary', {})
    utt = data.get('utterance_annotations', [])
    f   = {'chunk_id': chunk_id, '_json_loaded': True}

    # ── 4.1  Chunk-level derived features ───────────────────────────────────

    # Participation
    speaking_times = cs.get('speaking_time_seconds', {})
    if isinstance(speaking_times, dict) and speaking_times:
        f['gini_coefficient']      = compute_gini(list(speaking_times.values()))
        total_speak                = sum(speaking_times.values())
        max_speak                  = max(speaking_times.values())
        f['dominant_speaker_flag'] = int(total_speak > 0 and max_speak / total_speak > 0.50)
    else:
        f['gini_coefficient']      = None
        f['dominant_speaker_flag'] = None

    # Idea trajectory
    traj                = cs.get('idea_trajectory', 'ambiguous')
    f['is_convergent']  = int(traj == 'convergent')
    f['is_divergent']   = int(traj == 'divergent')
    f['is_procedural']  = int(traj == 'procedural')

    # Collective engagement
    ceg = cs.get('collective_engagement_level')
    f['collective_engagement_score'] = ceg if isinstance(ceg, (int, float)) else None

    # Cross-disciplinary bridging
    f['cross_disciplinary_bridging'] = int(cs.get('cross_disciplinary_bridging') == 'Yes')

    # Commitment signal
    f['commitment_signal'] = int(cs.get('explicit_commitment_signal') == 'Yes')

    # Artifact engagement
    f['screenshare_active']   = int(cs.get('screenshare_active') == 'Yes')
    f['artifact_interaction'] = int(cs.get('artifact_interaction') == 'Yes')

    # Problem specificity level (1–4 or NA→None)
    psl = cs.get('problem_specificity_level')
    f['problem_specificity_level'] = (
        int(psl) if psl not in (None, 'NA', 'null') and str(psl).isdigit() else None
    )

    # Decision crystallization level
    dcl = cs.get('decision_crystallization_level')
    f['decision_crystallization_level'] = (
        int(dcl) if dcl is not None and str(dcl).lstrip('-').isdigit() else None
    )

    # Ambition level
    ambition_map = {
        'not_applicable': 0, 'incremental': 1,
        'novel_application': 2, 'novel_combination': 3, 'paradigm_challenging': 4
    }
    amb = str(cs.get('ambition_level', 'not_applicable')).lower().replace(' ', '_')
    f['ambition_level_ordinal'] = ambition_map.get(amb, 0)
    f['is_novel_combination']   = int(amb in ('novel_combination', 'paradigm_challenging'))

    # Complementarity and shared vision
    f['explicit_complementarity'] = int(cs.get('explicit_complementarity_recognition') == 'Yes')
    f['skill_gap_identified']     = int(cs.get('skill_gap_identification') == 'Yes')
    f['shared_vision_present']    = int(cs.get('shared_vision_indicator') == 'Yes')
    f['pronoun_shift_occurred']   = int(cs.get('pronoun_shift_flag') == 'Yes')

    # Interpersonal signals
    f['personal_disclosure'] = int(cs.get('personal_disclosure') == 'Yes')

    lq = str(cs.get('laughter_quality', 'none')).lower()
    f['laughter_appreciative']    = int(lq == 'appreciative')
    f['laughter_shared_humor']    = int(lq == 'shared_humor')
    f['laughter_tension_release'] = int(lq == 'tension_release')
    f['any_laughter']             = int(lq != 'none')

    # Psychological safety — dissent response quality (1–3 or NA)
    drq = cs.get('dissent_response_quality')
    if drq in (None, 'NA', 'null', ''):
        f['dissent_response_quality']      = None
        f['dissent_was_present']           = 0
        f['dissent_response_exploratory']  = 0
    else:
        drq_int = int(drq) if str(drq).lstrip('-').isdigit() else None
        f['dissent_response_quality']      = drq_int
        f['dissent_was_present']           = 1
        f['dissent_response_exploratory']  = int(drq_int == 3)

    # Risk and ambition
    f['risk_acknowledgment_enthusiasm'] = int(
        cs.get('risk_acknowledgment_with_enthusiasm') == 'Yes')

    # Grant and context signals
    f['funding_awareness']  = int(cs.get('funding_awareness_signal') == 'Yes')
    f['prior_relationship'] = int(cs.get('prior_relationship_signal') == 'Yes')

    # Meeting process quality
    structure_map = {'unstructured': 0, 'loosely_structured': 1, 'structured': 2}
    ms = str(cs.get('meeting_structure_quality', 'unstructured')).lower().replace(' ', '_')
    f['meeting_structure_quality'] = structure_map.get(ms, 0)

    # ── 4.2  Utterance-level aggregated features ─────────────────────────────

    # Behavioral code counts (all 16 categories)
    for cat in ALL_CATEGORIES:
        f[f'num_{cat}'] = count_codes(utt, cat)

    # Inline idea_quality scores (Idea_Management, Integration_Practices, Knowledge_Sharing)
    for cat in ['Idea_Management', 'Integration_Practices', 'Knowledge_Sharing']:
        scores = _collect_idea_quality(utt, cat)
        f[f'mean_idea_quality_{cat}'] = float(np.mean(scores)) if scores else None
        f[f'pct_high_quality_{cat}']  = (sum(s >= 2 for s in scores) / len(scores)
                                          if scores else None)

    # Building vs. blocking ratio
    building = (
        count_codes(utt, 'Idea_Management',
                    subcodes=['proposes_new_idea', 'extends_existing_idea',
                              'combines_ideas', 'returns_to_earlier_idea']) +
        count_codes(utt, 'Integration_Practices') +
        count_codes(utt, 'Evaluation_Practices', subcodes=['supports_or_validates']) +
        count_codes(utt, 'Knowledge_Sharing')
    )
    blocking = (
        count_codes(utt, 'Evaluation_Practices',
                    subcodes=['critiques_or_challenges', 'devil_advocate', 'raises_concern']) +
        count_interruption_type(utt, 'competitive_interruption')
    )
    f['building_count']           = building
    f['blocking_count']           = blocking
    f['building_blocking_ratio']  = building / (blocking + EPSILON)
    f['pct_building']             = building / (building + blocking + EPSILON)

    # Individual code category counts
    f['num_future_oriented']      = count_codes(utt, 'Future_Oriented_Language')
    f['num_named_next_steps']     = count_codes(utt, 'Future_Oriented_Language',
                                                 subcodes=['named_next_step'])
    f['num_epistemic_bridging']   = count_codes(utt, 'Epistemic_Bridging')
    f['num_idea_attribution']     = count_codes(utt, 'Idea_Ownership_Attribution')
    f['num_complementarity']      = count_codes(utt, 'Complementarity_Articulation')
    f['num_role_anticipation']    = count_codes(utt, 'Role_Anticipation')
    f['num_broader_significance'] = count_codes(utt, 'Broader_Significance')
    f['num_novelty_signals']      = count_codes(utt, 'Idea_Novelty_Signal')
    f['num_scope_calibration']    = count_codes(utt, 'Coordination_Decision',
                                                 subcodes=['scope_calibration'])

    # Setback response
    num_setback_explores = count_codes(utt, 'Evaluation_Practices',
                                       subcodes=['setback_response_explores',
                                                 'setback_response_accepts_builds'])
    num_setback_defends  = count_codes(utt, 'Evaluation_Practices',
                                       subcodes=['setback_response_defends',
                                                 'setback_response_redirects'])
    f['num_setback_explores']     = num_setback_explores
    f['num_setback_defends']      = num_setback_defends
    f['explore_vs_defend_ratio']  = num_setback_explores / (num_setback_defends + EPSILON)

    # Pronoun framing
    num_joint       = count_codes(utt, 'Pronoun_Framing', subcodes=['joint_framing'])
    num_individual  = count_codes(utt, 'Pronoun_Framing', subcodes=['individual_framing'])
    total_framing   = num_joint + num_individual
    f['num_joint_framing']        = num_joint
    f['num_individual_framing']   = num_individual
    f['pct_joint_framing']        = num_joint / (total_framing + EPSILON)

    # Overall quality aggregates
    f['total_quality_score'] = sum_all_scores(utt)
    f['mean_quality_score']  = mean_all_scores(utt)
    f['pct_high_quality']    = pct_scores_gte(utt, threshold=1)

    # Interruption quality
    collab = count_interruption_type(utt, 'collaborative_completion')
    elab   = count_interruption_type(utt, 'elaborative_jump_in')
    compet = count_interruption_type(utt, 'competitive_interruption')
    f['num_collaborative_completions']  = collab
    f['num_elaborative_jumps']          = elab
    f['num_competitive_interruptions']  = compet
    f['interruption_quality_ratio']     = (collab + elab) / (compet + EPSILON)

    # ── 4.3  Multimodal signal aggregates ────────────────────────────────────

    f['pct_turns_distraction']         = pct_turns_with(utt, 'visible_off_screen_distraction', 'Yes')
    f['mean_distracted_count']         = mean_field(utt, 'distracted_participant_count')
    f['mean_nod_rate']                 = mean_field(utt, 'nod_count')
    f['pct_turns_shared_affect']       = pct_turns_with(utt, 'shared_affect', 'Yes')
    f['pct_turns_any_smile']           = pct_turns_with(utt, 'any_smile_other', 'Yes')
    f['pct_turns_audible_backchannel'] = pct_turns_with(utt, 'audible_backchannel', 'Yes')

    f['mean_vocal_enthusiasm']  = mean_field(utt, 'vocal_enthusiasm')
    f['pct_high_enthusiasm']    = pct_turns_with(
        [u for u in utt if u.get('vocal_enthusiasm') is not None],
        'vocal_enthusiasm',
        3  # vocal_enthusiasm >= 3
    )
    # Re-compute pct_high_enthusiasm correctly (>= 3)
    enth_vals = [coerce_numeric(u.get('vocal_enthusiasm')) for u in utt]
    enth_vals = [v for v in enth_vals if v is not None]
    f['pct_high_enthusiasm'] = (sum(v >= 3 for v in enth_vals) / len(enth_vals)
                                 if enth_vals else None)
    f['pct_hesitation']       = pct_turns_with(utt, 'hesitation_flag', 'Yes')

    chunk_rows.append(f)


chunk_df = pd.DataFrame(chunk_rows)

# Merge with registry metadata (conference_id, session_key, session_group,
# chunk_index, n_chunks_in_session, chunk_position, outcomes)
meta_cols = [
    'chunk_id', 'conference_id', 'session_key', 'session_group',
    'chunk_index', 'n_chunks_in_session', 'chunk_position',
    'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams',
    'outcome_has_funded_teams'
]
chunk_df = registry[meta_cols].merge(chunk_df, on='chunk_id', how='left')

print(f'chunk_df: {chunk_df.shape[0]:,} rows × {chunk_df.shape[1]} columns')
print(f'JSON loaded for {chunk_df["_json_loaded"].sum():,} chunks')
chunk_df[chunk_df['_json_loaded'].eq(True)].head(2)

chunk_df: 1,310 rows × 100 columns
JSON loaded for 1,286 chunks


,chunk_id,conference_id,session_key,session_group,chunk_index,n_chunks_in_session,chunk_position,num_teams,num_funded_teams,has_teams,...,interruption_quality_ratio,pct_turns_distraction,mean_distracted_count,mean_nod_rate,pct_turns_shared_affect,pct_turns_any_smile,pct_turns_audible_backchannel,mean_vocal_enthusiasm,pct_high_enthusiasm,pct_hesitation
0,2020NES__2020_11_05_NES_S5/Zoom_Meeting_11_5_2...,2020NES,2020_11_05_NES_S5/Zoom_Meeting_11_5_2020_12_17...,2020_11_05_NES_S5,0,1,whole,1.0,1.0,True,...,0.0,0.0,0.0,1.000000,0.0,0.000000,0.25,2.000000,0.000000,0.25
1,2020NES__2020_11_05_NES_S5/Zoom_Meeting_11_5_2...,2020NES,2020_11_05_NES_S5/Zoom_Meeting_11_5_2020_12_11...,2020_11_05_NES_S5,0,1,whole,1.0,1.0,True,...,0.0,0.0,0.0,0.666667,0.0,0.066667,0.00,2.133333,0.133333,0.00


## Step 5 — Compute responsiveness index (z-scored composite)

In [58]:
# Responsiveness index: z-score nod_rate, pct_shared_affect, pct_backchannel,
# then take the average.
# We compute per-chunk z-scores against the full corpus distribution.

responsive_feats = ['mean_nod_rate', 'pct_turns_shared_affect', 'pct_turns_audible_backchannel']

zscored = chunk_df[responsive_feats].apply(
    lambda col: (col - col.mean()) / col.std(ddof=1)
)
chunk_df['responsiveness_index'] = zscored.mean(axis=1)

print('Responsiveness index (mean, std):', chunk_df['responsiveness_index'].agg(['mean','std']).round(3).to_dict())

Responsiveness index (mean, std): {'mean': -0.007, 'std': 0.743}


## Step 6 — Session-level aggregation (Stage 4.4)

Aggregate chunk features into session-group–level features.  
Chunks are grouped by `session_group`; temporal segments use `chunk_position`.

In [59]:
def compute_enthusiasm_synchrony(utterances_all: list[dict]) -> float | None:
    """Energy matching index: mean pairwise Pearson r of per-turn vocal_enthusiasm across speakers.
    
    Groups turns by speaker, then for each pair of speakers computes Pearson r
    between their enthusiasm values on turns where both have spoken.
    Returns mean r across pairs, or None if fewer than 2 speakers.
    """
    from collections import defaultdict

    turn_enthusiasm: dict[str, list[float]] = defaultdict(list)
    for u in utterances_all:
        spk = u.get('speaker')
        enth = coerce_numeric(u.get('vocal_enthusiasm'))
        if spk and enth is not None:
            turn_enthusiasm[spk].append(enth)

    speakers = list(turn_enthusiasm.keys())
    if len(speakers) < 2:
        return None

    rs = []
    for i in range(len(speakers)):
        for j in range(i + 1, len(speakers)):
            a = turn_enthusiasm[speakers[i]]
            b = turn_enthusiasm[speakers[j]]
            n = min(len(a), len(b))
            if n < 3:
                continue
            try:
                r, _ = pearsonr(a[:n], b[:n])
                if not np.isnan(r):
                    rs.append(r)
            except Exception:
                pass

    return float(np.mean(rs)) if rs else None



def compute_parallel_monologue(utterances_all: list[dict]) -> float | None:
    """Parallel monologue index: fraction of new-idea proposals not building on prior speaker.
    
    A turn is a 'parallel monologue' if it contains proposes_new_idea AND the
    immediately preceding utterance by a *different* speaker also contains
    proposes_new_idea (i.e., both speakers are generating without responding).
    Returns None if no new-idea proposals exist.
    """
    new_idea_variants = set(CODE_NAME_VARIANTS['Idea_Management'])

    def has_new_idea(u: dict) -> bool:
        return any(
            c.get('code_name') in new_idea_variants and c.get('subcode') == 'proposes_new_idea'
            for c in u.get('codes', [])
        )

    proposals = [i for i, u in enumerate(utterances_all) if has_new_idea(u)]
    if not proposals:
        return None

    parallel_count = 0
    for idx in proposals:
        if idx == 0:
            continue
        curr_spk = utterances_all[idx].get('speaker')
        # Find the most recent utterance by a different speaker
        for prev_idx in range(idx - 1, -1, -1):
            prev_spk = utterances_all[prev_idx].get('speaker')
            if prev_spk != curr_spk:
                # Check if prior speaker's turn also proposed a new idea
                if not has_new_idea(utterances_all[prev_idx]):
                    parallel_count += 1  # prior speaker was NOT also proposing
                break

    return parallel_count / len(proposals)


print('Session-level helper functions defined.')

Session-level helper functions defined.


In [60]:
# Identify all numeric chunk-level features (exclude metadata and outcome columns)
META_COLS_SET = {
    'chunk_id', 'conference_id', 'session_key', 'session_group',
    'chunk_index', 'n_chunks_in_session', 'chunk_position',
    'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams',
    'outcome_has_funded_teams', '_json_loaded'
}
chunk_level_numeric_features = [
    c for c in chunk_df.columns
    if c not in META_COLS_SET and pd.api.types.is_numeric_dtype(chunk_df[c])
]

session_features_list = []

for session_group, grp in chunk_df.groupby('session_group'):
    sf = {'session_group': session_group}

    # Temporal segments (chunk_position assigned per recording within session_group)
    beg = grp[grp['chunk_position'] == 'beginning']
    mid = grp[grp['chunk_position'] == 'middle']
    end = grp[grp['chunk_position'] == 'end']
    # 'whole' sessions treated as both beginning and end
    whole = grp[grp['chunk_position'] == 'whole']
    beg   = pd.concat([beg, whole])
    end   = pd.concat([end, whole])

    n_chunks = len(grp)

    # ── Automatic aggregation: mean + temporal means + delta ─────────────────
    for feat in chunk_level_numeric_features:
        col_all  = pd.to_numeric(grp[feat],  errors='coerce')
        col_beg  = pd.to_numeric(beg[feat],  errors='coerce')
        col_mid  = pd.to_numeric(mid[feat],  errors='coerce')
        col_end  = pd.to_numeric(end[feat],  errors='coerce')

        sf[f'session_mean_{feat}']      = col_all.mean() if not col_all.dropna().empty else None
        sf[f'session_beginning_{feat}'] = col_beg.mean() if not col_beg.dropna().empty else None
        sf[f'session_middle_{feat}']    = col_mid.mean() if not col_mid.dropna().empty else None
        sf[f'session_end_{feat}']       = col_end.mean() if not col_end.dropna().empty else None

        mean_end = col_end.mean() if not col_end.dropna().empty else None
        mean_beg = col_beg.mean() if not col_beg.dropna().empty else None
        sf[f'session_delta_{feat}'] = (
            mean_end - mean_beg
            if mean_end is not None and mean_beg is not None
            else None
        )

    # ── Convergence and trajectory composites ────────────────────────────────
    sf['convergence_ratio_end']  = end['is_convergent'].mean()  if len(end) > 0 else None
    sf['divergent_ratio_beg']    = beg['is_divergent'].mean()   if len(beg) > 0 else None

    div_beg = beg['is_divergent'].mean()    if len(beg) > 0 else None
    div_end = end['is_divergent'].mean()    if len(end) > 0 else None
    con_beg = beg['is_convergent'].mean()   if len(beg) > 0 else None
    con_end = end['is_convergent'].mean()   if len(end) > 0 else None
    sf['diverge_converge_arc'] = int(
        all(v is not None for v in [div_beg, div_end, con_beg, con_end]) and
        div_beg > div_end and con_end > con_beg
    )

    # ── Commitment signal timing ─────────────────────────────────────────────
    commit_chunks = grp[grp['commitment_signal'] == 1]
    sf['any_commitment_signal']     = int(len(commit_chunks) > 0)
    sf['earliest_commitment_chunk'] = (
        int(commit_chunks['chunk_index'].min())
        if len(commit_chunks) > 0 else -1
    )
    sf['commitment_timing_normalized'] = (
        sf['earliest_commitment_chunk'] / (n_chunks - 1)
        if sf['any_commitment_signal'] and n_chunks > 1 else None
    )

    # ── Engagement trajectory ────────────────────────────────────────────────
    eng_end = end['collective_engagement_score'].dropna()
    eng_beg = beg['collective_engagement_score'].dropna()
    sf['engagement_trajectory'] = (
        eng_end.mean() - eng_beg.mean()
        if len(eng_end) > 0 and len(eng_beg) > 0 else None
    )

    # ── Cross-disciplinary bridging timing ───────────────────────────────────
    early = grp[grp['chunk_index'] <= 1]
    sf['early_bridging'] = int(early['cross_disciplinary_bridging'].any())

    # ── Participation trajectory ─────────────────────────────────────────────
    gini_end = end['gini_coefficient'].dropna()
    gini_beg = beg['gini_coefficient'].dropna()
    sf['gini_trajectory'] = (
        gini_end.mean() - gini_beg.mean()
        if len(gini_end) > 0 and len(gini_beg) > 0 else None
    )

    # ── Intellectual quality trajectories ────────────────────────────────────
    spec_all = pd.to_numeric(grp['problem_specificity_level'], errors='coerce').dropna()
    if len(spec_all) > 0:
        beg_spec = pd.to_numeric(beg['problem_specificity_level'], errors='coerce').dropna()
        end_spec = pd.to_numeric(end['problem_specificity_level'], errors='coerce').dropna()
        sf['problem_specificity_final'] = end_spec.mean() if len(end_spec) > 0 else None
        sf['problem_specificity_delta'] = (
            end_spec.mean() - beg_spec.mean()
            if len(beg_spec) > 0 and len(end_spec) > 0 else None
        )
    else:
        sf['problem_specificity_final'] = None
        sf['problem_specificity_delta'] = None

    dcl_end = pd.to_numeric(end['decision_crystallization_level'], errors='coerce').dropna()
    dcl_beg = pd.to_numeric(beg['decision_crystallization_level'], errors='coerce').dropna()
    sf['decision_crystallization_final'] = (
        float(end['decision_crystallization_level'].iloc[-1])
        if len(end) > 0 and pd.notna(end['decision_crystallization_level'].iloc[-1])
        else None
    )
    sf['decision_crystallization_delta'] = (
        dcl_end.mean() - dcl_beg.mean()
        if len(dcl_end) > 0 and len(dcl_beg) > 0 else None
    )

    sf['max_ambition_level']    = int(grp['ambition_level_ordinal'].max())
    sf['any_novel_combination'] = int(grp['is_novel_combination'].any())

    # ── Complementarity and shared vision ────────────────────────────────────
    sf['any_complementarity_recognized'] = int(grp['explicit_complementarity'].any())
    sf['any_skill_gap_identified']        = int(grp['skill_gap_identified'].any())
    sf['any_shared_vision']               = int(grp['shared_vision_present'].any())
    sf['pronoun_shift_occurred']          = int(grp['pronoun_shift_occurred'].any())

    shift_chunks = grp[grp['pronoun_shift_occurred'] == 1]
    sf['pronoun_shift_timing'] = (
        shift_chunks['chunk_index'].min() / (n_chunks - 1)
        if len(shift_chunks) > 0 and n_chunks > 1 else None
    )

    jft_end = end['pct_joint_framing'].dropna()
    jft_beg = beg['pct_joint_framing'].dropna()
    sf['joint_framing_trajectory'] = (
        jft_end.mean() - jft_beg.mean()
        if len(jft_end) > 0 and len(jft_beg) > 0 else None
    )

    # ── Building vs. blocking ────────────────────────────────────────────────
    sf['building_blocking_ratio_session'] = (
        grp['building_count'].sum() / (grp['blocking_count'].sum() + EPSILON)
    )
    sf['pct_building_session'] = (
        grp['building_count'].sum() /
        (grp['building_count'].sum() + grp['blocking_count'].sum() + EPSILON)
    )

    # ── Psychological safety index ───────────────────────────────────────────
    dissent_chunks = grp[grp['dissent_was_present'] == 1]
    sf['pct_dissent_exploratory'] = (
        dissent_chunks['dissent_response_exploratory'].mean()
        if len(dissent_chunks) > 0 else None
    )
    sf['mean_dissent_response_quality'] = (
        pd.to_numeric(dissent_chunks['dissent_response_quality'], errors='coerce').mean()
        if len(dissent_chunks) > 0 else None
    )

    psych_inputs = [
        sf.get('session_mean_interruption_quality_ratio'),
        sf.get('pct_dissent_exploratory'),
        sf.get('session_mean_pct_turns_audible_backchannel'),
        (1 - sf['session_mean_pct_hesitation'])
        if sf.get('session_mean_pct_hesitation') is not None else None,
    ]
    sf['psych_safety_index'] = mean_of_zscored(*psych_inputs)

    # ── Setback response ─────────────────────────────────────────────────────
    sf['explore_vs_defend_ratio_session'] = (
        grp['num_setback_explores'].sum() /
        (grp['num_setback_defends'].sum() + EPSILON)
    )

    # ── Affective and interpersonal signals ──────────────────────────────────
    sf['any_personal_disclosure']          = int(grp['personal_disclosure'].any())
    sf['pct_chunks_appreciative_laughter'] = grp['laughter_appreciative'].mean()
    sf['pct_chunks_any_laughter']          = grp['any_laughter'].mean()
    sf['any_risk_enthusiasm']              = int(grp['risk_acknowledgment_enthusiasm'].any())

    # ── Energy matching index ────────────────────────────────────────────────
    # Collect all utterances for this session_group from json_cache
    all_utt = []
    for cid in grp['chunk_id']:
        d = json_cache.get(cid, {})
        all_utt.extend(d.get('utterance_annotations', []))
    sf['energy_matching_index']   = compute_enthusiasm_synchrony(all_utt)
    sf['parallel_monologue_index'] = compute_parallel_monologue(all_utt)

    # ── Named next steps ─────────────────────────────────────────────────────
    final_two = grp.nlargest(2, 'chunk_index')
    sf['named_next_steps_count'] = int(final_two['num_named_next_steps'].sum())

    # ── Grant context signals ────────────────────────────────────────────────
    sf['any_funding_awareness']    = int(grp['funding_awareness'].any())
    sf['prior_relationship_present'] = int(grp['prior_relationship'].any())
    sf['any_broader_significance'] = int((grp['num_broader_significance'] > 0).any())

    # ── Meeting structure quality ────────────────────────────────────────────
    sf['mean_meeting_structure_quality'] = grp['meeting_structure_quality'].mean()
    sf['meeting_structure_final'] = (
        int(end['meeting_structure_quality'].iloc[-1])
        if len(end) > 0 and pd.notna(end['meeting_structure_quality'].iloc[-1])
        else None
    )

    # ── Absence flags ────────────────────────────────────────────────────────
    sf['no_convergence_flag'] = int(
        sf['decision_crystallization_final'] is not None and
        sf['decision_crystallization_final'] <= 2
    )
    sf['no_commitment_signal']           = int(not sf['any_commitment_signal'])
    sf['no_complementarity_recognition'] = int(not sf['any_complementarity_recognized'])

    unresolved = (dissent_chunks[dissent_chunks['dissent_response_quality'] == 1]
                  if len(dissent_chunks) > 0 else pd.DataFrame())
    sf['unresolved_tension_flag'] = int(len(unresolved) > 0)

    session_features_list.append(sf)


session_df = pd.DataFrame(session_features_list)
print(f'session_df: {session_df.shape[0]:,} sessions × {session_df.shape[1]} features')

/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is 

session_df: 162 sessions × 484 features


/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(a[:n], b[:n])
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_38564/4175815165.py:30: ConstantInputWarning: An input array is constant; the correlation coefficient is 

## Step 7 — Join with outcomes and conference (Stage 4.5)

In [61]:
# Session-group level outcomes and conference identity
outcomes_df = (
    registry
    .drop_duplicates('session_group')
    [['session_group', 'conference_id',
      'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams',
      'outcome_has_funded_teams']]
    .copy()
)

# Canonical outcome columns for modeling
outcomes_df['outcome_has_teams']          = outcomes_df['has_teams'].astype(float)
outcomes_df['outcome_has_funded_teams']   = outcomes_df['has_funded_teams'].astype(float)
outcomes_df['outcome_num_teams']          = outcomes_df['num_teams']
outcomes_df['outcome_num_funded_teams']   = outcomes_df['num_funded_teams']

features_df = session_df.merge(outcomes_df, on='session_group', how='left')

print(f'features_df: {features_df.shape[0]:,} sessions × {features_df.shape[1]} columns')
print('\nOutcome distribution:')
print(features_df[['outcome_has_teams', 'outcome_has_funded_teams']].apply(
    pd.to_numeric, errors='coerce').describe().round(3))

features_df: 162 sessions × 493 columns

Outcome distribution:
       outcome_has_teams  outcome_has_funded_teams
count            156.000                   156.000
mean               0.788                     0.436
std                0.410                     0.497
min                0.000                     0.000
25%                1.000                     0.000
50%                1.000                     0.000
75%                1.000                     1.000
max                1.000                     1.000


## Step 8 — Multicollinearity check via VIF (Stage 4.6)

In [62]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm.auto import tqdm

# Identify numeric modeling features (exclude metadata, outcomes, and IDs)
EXCLUDE_FROM_MODEL = {
    'session_group', 'conference_id',
    'outcome_has_teams', 'outcome_has_funded_teams',
    'outcome_num_teams', 'outcome_num_funded_teams',
    'num_teams', 'num_funded_teams', 'has_teams', 'has_funded_teams',
}

candidate_features = [
    c for c in features_df.columns
    if c not in EXCLUDE_FROM_MODEL
    and pd.api.types.is_numeric_dtype(features_df[c])
]

# Use only rows with complete data on ALL candidate features
vif_data = features_df[candidate_features].apply(pd.to_numeric, errors='coerce')
vif_data = vif_data.dropna(axis=1, thresh=int(len(vif_data) * 0.8))  # drop cols with >20% missing
vif_data = vif_data.dropna()                                          # drop rows with any remaining NA

print(f'VIF input: {vif_data.shape[0]} rows × {vif_data.shape[1]} features')

# Iteratively drop highest-VIF features until all VIF <= 10
vif_cols = list(vif_data.columns)
removed_for_vif = []

MAX_ITER = 100
pbar = tqdm(range(MAX_ITER), desc='VIF pruning', unit='iter')
for iteration in pbar:
    if len(vif_cols) < 2:
        pbar.set_postfix_str('fewer than 2 features left')
        break

    X = vif_data[vif_cols].values
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        vifs = [
            variance_inflation_factor(X, i)
            for i in range(X.shape[1])
        ]

    vif_series = pd.Series(vifs, index=vif_cols)
    max_vif = vif_series.max()
    pbar.set_postfix(max_vif=f'{max_vif:.2f}', n_features=len(vif_cols))

    if max_vif <= 10:
        print(f'\nVIF converged after {iteration} iterations. Max VIF = {max_vif:.2f}')
        break

    drop_feat = vif_series.idxmax()
    removed_for_vif.append((drop_feat, round(max_vif, 2)))
    vif_cols.remove(drop_feat)
else:
    print(f'VIF did not fully converge in {MAX_ITER} iterations.')

print(
    f'Features removed for VIF > 10 ({len(removed_for_vif)}):',
    [(f, v) for f, v in removed_for_vif[:10]],
    '...' if len(removed_for_vif) > 10 else ''
)
print(f'Retained modeling features: {len(vif_cols)}')

VIF input: 124 rows × 454 features


VIF pruning:   0%|          | 0/100 [00:00<?, ?iter/s]

VIF did not fully converge in 100 iterations.
Features removed for VIF > 10 (100): [('session_mean_gini_coefficient', np.float64(inf)), ('session_beginning_gini_coefficient', np.float64(inf)), ('session_middle_gini_coefficient', np.float64(inf)), ('session_end_gini_coefficient', np.float64(inf)), ('session_delta_gini_coefficient', np.float64(inf)), ('session_mean_dominant_speaker_flag', np.float64(inf)), ('session_middle_dominant_speaker_flag', np.float64(inf)), ('session_mean_is_convergent', np.float64(inf)), ('session_middle_is_convergent', np.float64(inf)), ('session_end_is_convergent', np.float64(inf))] ...
Retained modeling features: 354


In [63]:
# Final VIF report
if len(vif_cols) >= 2:
    X_final = vif_data[vif_cols].values
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        final_vifs = [variance_inflation_factor(X_final, i) for i in range(len(vif_cols))]
    vif_report = pd.DataFrame({'feature': vif_cols, 'VIF': final_vifs}).sort_values('VIF', ascending=False)
    print(vif_report.to_string(index=False))

                                               feature          VIF
                   session_middle_num_idea_attribution          inf
                  session_delta_num_individual_framing          inf
                    session_middle_total_quality_score          inf
                 session_beginning_total_quality_score          inf
                      session_mean_total_quality_score          inf
                       session_delta_pct_joint_framing          inf
                         session_end_pct_joint_framing          inf
                      session_middle_pct_joint_framing          inf
                   session_beginning_pct_joint_framing          inf
                        session_mean_pct_joint_framing          inf
                    session_end_num_individual_framing          inf
                  session_delta_num_epistemic_bridging          inf
                 session_middle_num_individual_framing          inf
              session_beginning_num_individual_f

## Step 9 — Build feature manifest and model-ready dataset

In [64]:
# Feature manifest: track which features were removed (VIF) and which are included
removed_set = {f for f, _ in removed_for_vif}

manifest_rows = []
for feat in candidate_features:
    manifest_rows.append({
        'feature_name':       feat,
        'included_in_model':  feat in vif_cols,
        'removed_reason':     'high_VIF' if feat in removed_set else
                              ('missing_data' if feat not in vif_data.columns else ''),
    })

manifest_df = pd.DataFrame(manifest_rows)
print(manifest_df['included_in_model'].value_counts().to_string())

# Model-ready features: session features post-VIF filter, joined with outcomes
outcome_cols = [
    'session_group', 'conference_id',
    'outcome_has_teams', 'outcome_has_funded_teams',
    'outcome_num_teams', 'outcome_num_funded_teams'
]
model_ready_df = features_df[outcome_cols + vif_cols].copy()

print(f'\nmodel_ready_df: {model_ready_df.shape[0]} sessions × {model_ready_df.shape[1]} columns')

included_in_model
True     354
False    129

model_ready_df: 162 sessions × 360 columns


## Step 10 — Save outputs (Stage 4.7)

In [67]:
# Stage 10 - save outputs
chunk_out = DATA_DIR / 'chunk_features.parquet'
session_out = DATA_DIR / 'session_features.parquet'

def save_table(df, parquet_path):
    csv_path = parquet_path.with_suffix('.csv')
    try:
        df.to_parquet(parquet_path, index=False)
        print(f'Saved {len(df):,} rows -> {parquet_path}')
    except Exception as e:
        print(f'Parquet save failed for {parquet_path.name}: {type(e).__name__}: {e}')
        df.to_csv(csv_path, index=False)
        print(f'Saved {len(df):,} rows -> {csv_path}')

save_table(chunk_df, chunk_out)
save_table(session_df, session_out)


Parquet save failed for chunk_features.parquet: ArrowKeyError: A type extension with name pandas.period already defined
Saved 1,310 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/chunk_features.csv
Parquet save failed for session_features.parquet: ArrowKeyError: A type extension with name pandas.period already defined
Saved 162 rows -> /Users/maxchalekson/Projects/NICO-Research/gemini_data_analysis/analysis_v2/data/session_features.csv


In [70]:
def load_table(stem: str) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'

    print(f'Looking for: {parquet_path}')
    print(f'Looking for: {csv_path}')

    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f'Falling back to CSV for {stem}: {type(e).__name__}: {e}')

    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(f'Could not find {parquet_path} or {csv_path}')


## Step 11 — Spot-check

In [68]:
print('── Chunk-level feature summary (sample numeric cols) ────────────────────')
sample_chunk_cols = [
    'gini_coefficient', 'dominant_speaker_flag', 'is_convergent', 'is_divergent',
    'collective_engagement_score', 'building_blocking_ratio', 'pct_joint_framing',
    'mean_nod_rate', 'responsiveness_index', 'mean_vocal_enthusiasm'
]
print(chunk_df[[c for c in sample_chunk_cols if c in chunk_df.columns]].describe().round(3))

── Chunk-level feature summary (sample numeric cols) ────────────────────
       gini_coefficient  dominant_speaker_flag  is_convergent  is_divergent  \
count          1278.000               1278.000       1286.000      1286.000   
mean              0.349                  0.160          0.186         0.579   
std               0.127                  0.366          0.389         0.494   
min               0.000                  0.000          0.000         0.000   
25%               0.272                  0.000          0.000         0.000   
50%               0.346                  0.000          0.000         1.000   
75%               0.433                  0.000          0.000         1.000   
max               0.723                  1.000          1.000         1.000   

       collective_engagement_score  building_blocking_ratio  \
count                     1286.000             1.286000e+03   
mean                         2.977             4.278388e+06   
std                      

In [69]:
print('── Session-level outcome × key feature (grouped mean) ───────────────────')
key_session_feats = [
    'outcome_has_funded_teams',
    'session_mean_gini_coefficient',
    'session_mean_building_blocking_ratio',
    'session_mean_collective_engagement_score',
    'any_commitment_signal',
    'any_shared_vision',
    'session_mean_pct_joint_framing',
]
avail = [c for c in key_session_feats if c in features_df.columns]
grp_means = (
    features_df[avail]
    .apply(pd.to_numeric, errors='coerce')
    .groupby(features_df['outcome_has_funded_teams'].apply(pd.to_numeric, errors='coerce'))
    .mean()
    .round(3)
)
print(grp_means)

── Session-level outcome × key feature (grouped mean) ───────────────────
                          outcome_has_funded_teams  \
outcome_has_funded_teams                             
0.0                                            0.0   
1.0                                            1.0   

                          session_mean_gini_coefficient  \
outcome_has_funded_teams                                  
0.0                                               0.340   
1.0                                               0.363   

                          session_mean_building_blocking_ratio  \
outcome_has_funded_teams                                         
0.0                                                4275351.958   
1.0                                                4340111.813   

                          session_mean_collective_engagement_score  \
outcome_has_funded_teams                                             
0.0                                                          2.957 